# Agricultural NDVI Sensor Data Analysis

This project aim is to engineer and analyse the agricultural NDVI sensor data to evaluate:

- data quality issues,
- preprocessing steps for cleaning
- reaasoning behind those steps
- and overall quality before any modeling or time series forecasting.

The objective is to construct a trusted analytical dataset while preserving important operational anomalies separately for governance and reliability analysis.

In [11]:
# ============================================================
# IMPORTS and SETUP
# ============================================================

import sys
import os
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath(".."))
from functions.tools import configure_notebook_display

configure_notebook_display()

In [12]:
# ============================================================
# LOADING DATASETS
# ============================================================

from functions.tools import load_raw_datasets
df_meta, df_readings = load_raw_datasets()

In [13]:
# ============================================================
# INITIAL METADATA OVERVIEW
# ============================================================

print("Metadata Shape:", df_meta.shape)
display(df_meta.sample(5))

df_meta.info()

Metadata Shape: (28, 5)


,parcel_id,mill_id,crop_type,sowing_date,area_hectares
2,PARCEL_003,MILL_NORTH,soybean,2026-02-13,5.35
21,PARCEL_022,MILL_EAST,sugarcane,2026-01-14,8.04
17,PARCEL_018,MILL_SOUTH,sugarcane,2026-01-11,5.82
1,PARCEL_002,MILL_SOUTH,sugarcane,2026-01-16,8.97
11,PARCEL_012,MILL_WEST,sugarcane,2026-03-01,6.85


<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   parcel_id      28 non-null     str    
 1   mill_id        28 non-null     str    
 2   crop_type      28 non-null     str    
 3   sowing_date    28 non-null     str    
 4   area_hectares  28 non-null     float64
dtypes: float64(1), str(4)
memory usage: 1.2 KB


The metadata shows no immediate signs of missing entries.

One problem her is that sowing_date is stored as a string instead of a datetime field. This could be because of multiple sources, manual entries and want to keep the date injection flexible.

In [14]:
display(df_meta.describe(include="all"))

,parcel_id,mill_id,crop_type,sowing_date,area_hectares
count,28,28,28,28,28.000000
unique,28,4,3,23,NaN
top,PARCEL_001,MILL_NORTH,sugarcane,2026-01-16,NaN
freq,1,9,20,2,NaN
mean,NaN,NaN,NaN,NaN,5.787143
std,NaN,NaN,NaN,NaN,2.608158
min,NaN,NaN,NaN,NaN,1.550000
25%,NaN,NaN,NaN,NaN,3.457500
50%,NaN,NaN,NaN,NaN,5.550000
75%,NaN,NaN,NaN,NaN,7.800000


28 unique parcel ids in the metadata with 4 unique mills, 3 unique crop types.

While also area_hectares is looking in a realistic range.

In [15]:
# ============================================================
# INITIAL READINGS OVERVIEW
# ============================================================

print("Readings Shape:", df_readings.shape)

display(df_readings.sample(5))

df_readings.info()

Readings Shape: (3447, 6)


,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
1675,PARCEL_006,2026-04-02,0.780,30.5,0.0,OK
651,PARCEL_002,06/03/2026,0.626,34.6,7.2,OK
1650,PARCEL_017,2026-01-11,0.111,26.5,0.0,OK
709,PARCEL_025,2026-01-24,0.169,21.8,0.0,OK
203,PARCEL_009,2026-03-10,0.242,25.4,0.0,OK


<class 'pandas.DataFrame'>
RangeIndex: 3447 entries, 0 to 3446
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   parcel_id      3447 non-null   str    
 1   date           3447 non-null   str    
 2   ndvi_value     3447 non-null   float64
 3   temperature_c  3447 non-null   float64
 4   rainfall_mm    3447 non-null   float64
 5   sensor_status  3310 non-null   str    
dtypes: float64(3), str(3)
memory usage: 161.7 KB


The Readings dataset shows consistency in no missing rows except in the sensor_status.

Again, just like metadata, readings date entries is string as well instead of datetime.

In [16]:
display(df_readings.describe(include="all"))

,parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status
count,3447,3447,3447.000000,3447.000000,3447.000000,3310
unique,27,440,NaN,NaN,NaN,7
top,PARCEL_007,2026-04-05,NaN,NaN,NaN,OK
freq,142,22,NaN,NaN,NaN,2890
mean,NaN,NaN,0.464235,25.389614,0.831303,NaN
std,NaN,NaN,0.349201,5.744316,2.046716,NaN
min,NaN,NaN,-1.976000,15.100000,0.000000,NaN
25%,NaN,NaN,0.278000,20.600000,0.000000,NaN
50%,NaN,NaN,0.502000,25.400000,0.000000,NaN
75%,NaN,NaN,0.653000,30.400000,0.000000,NaN


One small finding here is that one of the parcel id is not showing up in the readings, as 28 unique parcel ids are there in metadata and only 27 unique parcel ids in the readings.

NDVI is immediately failing physical plausibility checks with values extending below -1 and above 1 with `-1.97` and `1.99` min and max values respectively. These readings are likely representing corrupted or degraded sensing behavior and not a real vegetation conditions.

Temperature and rainfall values are looking surprisingly stable and realistic. This suggests that temp and rainfall are coming from a centralised and more reliable source while NDVI is coming from noiser source. At this stage, the environmental and metadata layers are appearing significantly more trustworthy than the NDVI sensing layer.

After manually checking the date column in the csvs for both of the dataset, multiple date formats are coexisting instead of following a single standardized format.

They look like:

`2026-01-27`

`16/05/2026`

`04-Jan-2026`


In [17]:
# ============================================================
# DUPLICATES CHECKING
# ============================================================

meta_duplicates = df_meta.duplicated().sum()
reading_duplicates = df_readings.duplicated().sum()

print(f"Metadata Exact Duplicates: {meta_duplicates}")
print(f"Readings Exact Duplicates: {reading_duplicates}")

Metadata Exact Duplicates: 0
Readings Exact Duplicates: 0


No exact duplicate rows are appearing in either dataset.

In [18]:
# ============================================================
# CATEGORY DISTRIBUTIONS
# ============================================================

print("Crop Type Distribution in Metadata")
display(df_meta["crop_type"].value_counts(dropna=False))

print("\nMill Distribution in Metadata")
display(df_meta["mill_id"].value_counts(dropna=False))

print("\nSensor Status Distribution in Readings")
display(df_readings["sensor_status"].value_counts(dropna=False))

Crop Type Distribution in Metadata


crop_type
sugarcane    20
soybean       6
wheat         2
Name: count, dtype: int64


Mill Distribution in Metadata


mill_id
MILL_NORTH    9
MILL_EAST     9
MILL_SOUTH    5
MILL_WEST     5
Name: count, dtype: int64


Sensor Status Distribution in Readings


sensor_status
OK       2890
NaN       137
Error      90
ERROR      80
error      76
ok         64
OK         57
 OK        53
Name: count, dtype: int64

Sugarcane is dominating the crop distribution, while wheat has very limited representation.

The imbalance in the crop categories count needs to scaled for any crop-level modeling operations.

Mill distribution is staying relatively balanced across regions.

The sensor_status column is also showing strong inconsistency through casing, whitespace errors and 137 missing entries. This column is reflecting weak validation and injestion rules set in the downstream pipeline.

In [19]:
# ============================================================
# INVALID NDVI AND SENSOR RELIABILITY RELATIONSHIP
# ============================================================

# Invalid NDVI rows
invalid_ndvi = df_readings[
    (df_readings["ndvi_value"] < -1) |
    (df_readings["ndvi_value"] > 1)
    ]

# Rows where status is NOT any form of OK
invalid_and_error_status = invalid_ndvi[
    ~invalid_ndvi["sensor_status"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("ok")
    ]

total_invalid_ndvi = len(invalid_ndvi)
invalid_with_error_status = len(invalid_and_error_status)

percentage = (invalid_with_error_status / total_invalid_ndvi) * 100

print(f"Total Invalid NDVI Rows: {total_invalid_ndvi}")
print(f"Invalid NDVI Rows with Non-OK Status: {invalid_with_error_status}")
print(f"Percentage: {percentage:.2f}%")

Total Invalid NDVI Rows: 104
Invalid NDVI Rows with Non-OK Status: 104
Percentage: 100.00%


All invalid NDVI readings are occurring alongside non-ok sensor states. Resulting in a complete 100% overlap between impossible vegetation values and degraded operational statuses.

This is becoming one of the strongest signals in the dataset so far. The invalid NDVI values no longer appear random. They are strongly behaving like operational sensor corruption linked directly to unhealthy sensing conditions.

We could also verify in what temporal phase these sensors fail. By applying a time series analysis for failures of the sensors.

In [ ]:
issue_tracker = pd.DataFrame({

    "column": [
        "parcel_id",
        "crop_type",
        "sowing_date",
        "date",
        "ndvi_value",
        "sensor_status"
    ],

    "issue_identified": [

        "Parcel mismatch between metadata and readings",
        "Imbalanced crop distribution",
        "Stored as string instead of datetime",
        "Multiple date formats within the same column",
        "NDVI values outside valid biological range [-1, 1]",
        "Dirty categorical labels and missing statuses"
    ],

    "prevalence": [

        "28 in metadata, 27 in readings",
        "Sugarcane dominates by ~ 70% of all parcels",
        "Entire column",
        "3 types of formats detected",
        "104 invalid records detected",
        "Whitespace, casing issues and 137 missing values detected"
    ],

    "severity": [

        "High",
        "Low",
        "Medium",
        "High",
        "High",
        "Medium"
    ]
})

display(issue_tracker)

,column,issue_identified,prevalence,severity
0,parcel_id,Parcel mismatch between metadata and readings,"28 in metadata, 27 in readings",High
1,crop_type,Imbalanced crop distribution,Sugarcane dominates by ~ 70% of all parcels,Low
2,sowing_date,Stored as string instead of datetime,Entire column,Medium
3,date,Multiple date formats within the same column,3 types of formats detected,High
4,ndvi_value,"NDVI values outside valid biological range [-1, 1]",104 invalid records detected,High
5,sensor_status,Dirty categorical labels and missing statuses,"Whitespace, casing issues and 137 missing values detected",Medium
